In [28]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
# Import the wcv collision data csv file into a dataframe
df = pd.read_csv('WCV_data/datasets/WCV Collision Data.csv')
# Remove spaces and replace from remaining column names
df.columns = df.columns.str.replace(' ', '')

In [17]:
df.columns

Index(['OrganizationName', 'CaseNumber', 'PatientID', 'CommonSpeciesName',
       'DateAdmitted', 'CircumstancesofRescue', 'RescueState',
       'RescueJuristiction', 'RescueAddress', 'OtherRescueInformation',
       'Latitude', 'Longitude', 'Elevation', 'Disposition'],
      dtype='object')

In [18]:
df['RescueJuristiction'].unique()

array(['New Kent County', 'Richmond (City)', 'Bedford County',
       'Waynesboro (City)', 'Louisa County', 'Augusta County',
       'Fluvanna County', 'Rockbridge County', 'Orange County',
       'Charlottesville (City)', 'Harrisonburg (City)',
       'Albemarle County', 'Montgomery County', 'Middlesex County',
       'Nottoway County', 'Caroline County', 'Staunton (City)',
       'Chesterfield County', 'Spotsylvania County', 'Madison County',
       'Amherst County', 'Buckingham County', 'Henrico County',
       'Rockingham County', 'Campbell County', 'Nelson County',
       'Prince George County', 'Frederick County', 'Page County',
       'Essex County', 'Shenandoah County', 'Hanover County',
       'Giles County', 'Charlotte County', 'Lynchburg (City)',
       'Culpeper County', 'Pittsylvania County', 'Prince Edward County',
       'Roanoke County', 'Halifax County', 'Roanoke (City)',
       'Fairfax County', 'Virginia Beach (City)', 'Goochland County',
       'Fauquier County', 'G

In [29]:
# Drop rows where lat long is null
df = df.dropna(subset=['Latitude', 'Longitude'])

In [20]:
df[['Latitude', 'Longitude', 'RescueJuristiction']]

,Latitude,Longitude,RescueJuristiction
0,37.510148,-77.190808,New Kent County
1,37.508531,-77.485949,Richmond (City)
3,38.091202,-78.870465,Waynesboro (City)
4,38.022446,-78.005264,Louisa County
5,38.160019,-78.847099,Augusta County
...,...,...,...
1734,38.133640,-78.955917,Augusta County
1735,37.371113,-79.017906,Campbell County
1736,38.265703,-77.671162,Spotsylvania County
1737,38.449671,-78.849119,Harrisonburg (City)


In [24]:
us_county_shapes = gpd.read_file('./tl_2025_us_county/tl_2025_us_county.shp')
va_county_shapes = us_county_shapes[us_county_shapes["STATEFP"] == "51"]

In [ ]:
# Print the list of counties to validate that this is virginia
va_county_shapes["NAMELSAD"].to_list()

In [30]:
geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
df_geo = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# Ensure va_county_shapes has the same CRS                                             
va_county_shapes = va_county_shapes.to_crs("EPSG:4326")

# Spatial join - this matches each point to the polygon it falls within                
df_with_jurisdiction = gpd.sjoin(df_geo, va_county_shapes, how="left",                 
predicate="within")

In [33]:
df_with_jurisdiction[['Latitude', 'Longitude', 'RescueJuristiction', 'NAMELSAD']] 

,Latitude,Longitude,RescueJuristiction,NAMELSAD
0,37.510148,-77.190808,New Kent County,New Kent County
1,37.508531,-77.485949,Richmond (City),Richmond city
3,38.091202,-78.870465,Waynesboro (City),Waynesboro city
4,38.022446,-78.005264,Louisa County,Louisa County
5,38.160019,-78.847099,Augusta County,Augusta County
...,...,...,...,...
1734,38.133640,-78.955917,Augusta County,Augusta County
1735,37.371113,-79.017906,Campbell County,Campbell County
1736,38.265703,-77.671162,Spotsylvania County,Spotsylvania County
1737,38.449671,-78.849119,Harrisonburg (City),Harrisonburg city
